In [1]:
import os
import glob
import pickle
import prody
import json
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
import multiprocessing as mp
from collections import defaultdict


# --- Constant Definitions ---
RESTYPE_3_TO_1 = {
    "ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C", "GLN": "Q",
    "GLU": "E", "GLY": "G", "HIS": "H", "ILE": "I", "LEU": "L", "LYS": "K",
    "MET": "M", "PHE": "F", "PRO": "P", "SER": "S", "THR": "T", "TRP": "W",
    "TYR": "Y", "VAL": "V",
}
STANDARD_AMINO_ACIDS_3 = set(RESTYPE_3_TO_1.keys())

ATOM_ORDER = {
    "N": 0, "CA": 1, "C": 2, "CB": 3, "O": 4, "CG": 5, "CG1": 6, "CG2": 7,
    "OG": 8, "OG1": 9, "SG": 10, "CD": 11, "CD1": 12, "CD2": 13, "ND1": 14,
    "ND2": 15, "OD1": 16, "OD2": 17, "SD": 18, "CE": 19, "CE1": 20, "CE2": 21,
    "CE3": 22, "NE": 23, "NE1": 24, "NE2": 25, "OE1": 26, "OE2": 27, "CH2": 28,
    "NH1": 29, "NH2": 30, "OH": 31, "CZ": 32, "CZ2": 33, "CZ3": 34, "NZ": 35,
    "OXT": 36,
}
MAX_ATOMS = len(ATOM_ORDER)

# --- Path Configuration ---
PDB_DIR = "/data/CPSea/CPCore/CPCore/CPCore_pdb/CPCore_pdb/"
TSV_FILE = "/data/CPSea/CPCore/CPCore/CPCore_properties/CPCore_Basic.tsv"
OUTPUT_DIR = "/data/CPSea/CPCore/CPCore_processed_pkls/"

os.makedirs(OUTPUT_DIR, exist_ok=True)
prody.confProDy(verbosity='none')

PKL_DIR = OUTPUT_DIR
WORK_DIR = "/data/CPSea/CPCore/split_workflow/"
os.makedirs(WORK_DIR, exist_ok=True)

FASTA_FILE = os.path.join(WORK_DIR, "all_sequences.fasta")
SOURCE_MAP_FILE = os.path.join(WORK_DIR, "seq_id_to_source_map.json")
CLUSTER_FILE = os.path.join(WORK_DIR, "clusterRes_cluster.tsv")
FINAL_SPLIT_JSON = os.path.join(WORK_DIR, "data_splits.json")

/home/qiuyk/miniconda3/envs/LiUniIF/lib/python3.10/site-packages/prody/utilities/misctools.py:424: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [ ]:
def extract_chain_features(chain_selection: prody.Selection, chain_id: str):
    residues = list(chain_selection.getHierView().iterResidues())
    num_residues = len(residues)

    xyz_37 = np.zeros((num_residues, MAX_ATOMS, 3), dtype=np.float32)
    mask_37 = np.zeros((num_residues, MAX_ATOMS), dtype=bool)
    res_indices = np.zeros(num_residues, dtype=np.int32)
    sequence_list = []

    for i, res in enumerate(residues):
        # Strictly record the residue numbers of the original file
        res_indices[i] = res.getResnum()
        sequence_list.append(RESTYPE_3_TO_1.get(res.getResname(), 'X'))

        for atom in res:
            atom_name = atom.getName()
            if atom_name in ATOM_ORDER:
                atom_idx = ATOM_ORDER[atom_name]
                xyz_37[i, atom_idx] = atom.getCoords()
                mask_37[i, atom_idx] = True

    sequence = "".join(sequence_list)
    backbone_mask = (mask_37[:, ATOM_ORDER['N']] &
                     mask_37[:, ATOM_ORDER['CA']] &
                     mask_37[:, ATOM_ORDER['C']] &
                     mask_37[:, ATOM_ORDER['O']])

    return {
        'xyz_37': xyz_37,
        'xyz_37_mask': mask_37,
        'seq': sequence,
        'R_idx': res_indices,
        'chain_id': chain_id,
        'mask': backbone_mask
    }

def check_chain_validity(chain_selection: prody.Selection):
    if chain_selection is None or chain_selection.numAtoms() == 0:
        return False, []

    residues = list(chain_selection.getHierView().iterResidues())

    for res in residues:
        # 1. Check if it contains non-standard amino acids
        if res.getResname() not in STANDARD_AMINO_ACIDS_3:
            return False, []

        # 2. Check if the main atoms are missing
        if (res.getAtom('N') is None or res.getAtom('CA') is None or
                res.getAtom('C') is None or res.getAtom('O') is None):
            return False, []

    return True, residues

def process_complex(pdb_path: str, tsv_metadata: dict):
    base_name = os.path.basename(pdb_path)
    complex_id = base_name.replace(".pdb.gz", "").replace(".pdb", "")
    output_pkl_path = os.path.join(OUTPUT_DIR, f"{complex_id}.pkl")

    if os.path.exists(output_pkl_path):
        return "Already Processed"

    try:
        structure = prody.parsePDB(pdb_path)
        if structure is None:
            return "Parse Failed"

        # Only select "protein", automatically filter out water molecules and non-atomic substances
        protein_struct = structure.select('protein')
        if protein_struct is None:
            return "No Protein"

        # Extract L-chain (cyclic peptide)
        chain_L_sel = protein_struct.select('chain L')
        is_L_valid, residues_L = check_chain_validity(chain_L_sel)
        if not is_L_valid:
            return "Invalid L Chain (Non-standard AA or missing backbone)"

        # Extract R chain (receptor)
        chain_R_sel = protein_struct.select('chain R')
        is_R_valid, residues_R = check_chain_validity(chain_R_sel)
        if not is_R_valid:
            return "Invalid R Chain (Non-standard AA or missing backbone)"

        # Check for receptor length constraints (< 6000)
        if len(residues_R) >= 6000:
            return "R Chain Too Long (>=6000)"

        feature_L = extract_chain_features(chain_L_sel, 'L')
        feature_R = extract_chain_features(chain_R_sel, 'R')

        meta_info = tsv_metadata.get(complex_id, {})

        final_data = {
            'complex_id': complex_id,
            'peptide_L': feature_L,
            'receptor_R': feature_R,
            'metadata': meta_info
        }

        with open(output_pkl_path, 'wb') as f:
            pickle.dump(final_data, f)

        return "Success"

    except Exception as e:
        return f"Error: {str(e)}"

def process_wrapper(args):
    pdb_path, tsv_metadata = args
    return process_complex(pdb_path, tsv_metadata)

def main():
    print("Load tsv attribute file...")
    try:
        df_meta = pd.read_csv(TSV_FILE, sep='\t')
        # Set the "id" column as the index and convert it into a nested dictionary {id: {length: ...} , cyclic_type: ... }}
        tsv_metadata = df_meta.set_index('id').to_dict(orient='index')
        print(f"Successfully loaded {len(tsv_metadata)} pieces of metadata.")
    except Exception as e:
        print(f"Failed to read tsv file: {e}")
        return

    pdb_files = glob.glob(os.path.join(PDB_DIR, '*.pdb.gz'))
    if not pdb_files:
        print(f"No .pdb.gz files were found in {PDB_DIR}!")
        return

    print(f"A total of {len(pdb_files)} PDB files have been found and are ready to be processed...")

    num_cores = max(1, mp.cpu_count() - 1)
    print(f"Starting {num_cores} child processes to accelerate the extraction...")

    stats = {
        "Success": 0,
        "Already Processed": 0,
        "Invalid L Chain (Non-standard AA or missing backbone)": 0,
        "Invalid R Chain (Non-standard AA or missing backbone)": 0,
        "R Chain Too Long (>=6000)": 0,
        "Parse Failed": 0,
        "No Protein": 0,
    }

    task_args = [(path, tsv_metadata) for path in pdb_files]

    with mp.Pool(processes=num_cores) as pool:
        for status in tqdm(pool.imap_unordered(process_wrapper, task_args), total=len(task_args), desc="Processing CPCore"):
            if status.startswith("Error:"):
                stats[status] = stats.get(status, 0) + 1
            else:
                stats[status] = stats.get(status, 0) + 1

    print("\n" + "="*40)
    print("Processing completed! The statistics are as follows:")
    for k, v in stats.items():
        if v > 0:
            print(f"{k}: {v}")
    print(f"The Pickle file has been saved to: {OUTPUT_DIR}")
    print("="*40)

if __name__ == "__main__":
    main()

In [ ]:
print("--- Step 0: Setup ---")

SEED = 42
random.seed(SEED)

def extract_all_sequences():
    print("\n--- Step 1: Extracting all sequences from .pkl files... ---")
    
    pkl_files = glob.glob(os.path.join(PKL_DIR, "*.pkl"))
    print(f"Found {len(pkl_files)} .pkl files.")
    
    total_chains = 0
    seq_id_to_source_map = {}
    
    with open(FASTA_FILE, 'w') as f_out:
        for pkl_file in tqdm(pkl_files, desc="Processing CPCore"):
            try:
                with open(pkl_file, 'rb') as f_in:
                    data = pickle.load(f_in)
                
                complex_id = data['complex_id']

                l_seq = data['peptide_L']['seq']
                l_id = f"CPCore-{complex_id}-L"
                f_out.write(f">{l_id}\n{l_seq}\n")
                seq_id_to_source_map[l_id] = {
                    "dataset": "CPCore",
                    "complex_id": complex_id,
                    "chain_type": "L"
                }
                total_chains += 1

                r_seq = data['receptor_R']['seq']
                r_id = f"CPCore-{complex_id}-R"
                f_out.write(f">{r_id}\n{r_seq}\n")
                seq_id_to_source_map[r_id] = {
                    "dataset": "CPCore",
                    "complex_id": complex_id,
                    "chain_type": "R"
                }
                total_chains += 1
                
            except Exception as e:
                print(f"Error processing file {pkl_file}: {e}")

    with open(SOURCE_MAP_FILE, 'w') as f:
        json.dump(seq_id_to_source_map, f, indent=4)

    print(f"✅ Extraction completed! A total of {total_chains} single-chain sequences were extracted.")
    print(f"FASTA saved to: {FASTA_FILE}")
    print(f"Save the Source Map to: {SOURCE_MAP_FILE}")

extract_all_sequences()

In [ ]:
def mmseqs_instructions():
    print("\n--- Step 2: Running MMseqs2 for global sequence clustering... ---")
    print("Please copy the following command and run it in your Linux terminal:")
    
    cmd = f"mmseqs easy-cluster {FASTA_FILE} {os.path.join(WORK_DIR, 'clusterRes')} {os.path.join(WORK_DIR, 'tmp')} --min-seq-id 0.3 -c 0.8 --cov-mode 1 --threads 8"
    print(f"\n\033[93m{cmd}\033[0m\n")
    print(f"After the operation is completed, {CLUSTER_FILE} will be generated.")

mmseqs_instructions()

In [ ]:
import os
import json
import random
from collections import defaultdict

def generate_final_splits():
    print("\n--- Step 3: Generating custom data splits (Train / Val / Test) ---")
    
    if not os.path.exists(CLUSTER_FILE):
        print(f"Error: Cluster result file {CLUSTER_FILE} not found. Please run MMseqs2 first!")
        return

    # 1. Load mapping files
    print("Step 3.1: Loading cluster results and source map...")
    with open(SOURCE_MAP_FILE, 'r') as f:
        seq_id_to_source = json.load(f)
        
    clusters = defaultdict(list)
    chain_to_rep = {}
    with open(CLUSTER_FILE, 'r') as f:
        for line in f:
            representative, member = line.strip().split('\t')
            clusters[representative].append(member)
            chain_to_rep[member] = representative

    # 2. Identify orphan clusters (<5) and healthy clusters (>=5)
    print("Step 3.2: Filtering orphan clusters (< 5 members)...")
    orphan_reps = []
    healthy_reps = []
    
    for rep, members in clusters.items():
        if len(members) < 5:
            orphan_reps.append(rep)
        else:
            healthy_reps.append(rep)
            
    print(f"Total clusters: {len(clusters)}")
    print(f"-> Orphan clusters (<5): {len(orphan_reps)} (All assigned to Train)")
    print(f"-> Healthy clusters (>=5): {len(healthy_reps)} (Used for Test/Val/Train split)")

    # 3. Custom Splitting Logic (Train, Val, Test)
    TEST_RATIO = 0.05  
    VAL_RATIO = 0.05
    print(f"Step 3.3: Assigning {TEST_RATIO*100}% to Test, {VAL_RATIO*100}% to Validation, and the rest to Train...")
    
    random.seed(3407)
    random.shuffle(healthy_reps)
    
    num_test = int(len(healthy_reps) * TEST_RATIO)
    num_val = int(len(healthy_reps) * VAL_RATIO)
    
    test_reps = healthy_reps[:num_test]
    val_reps = healthy_reps[num_test:num_test + num_val]
    train_reps = orphan_reps + healthy_reps[num_test + num_val:]
    
    split_reps = {
        'test': test_reps,
        'validation': val_reps,
        'train': train_reps
    }
    
    # Build mapping: chain -> assigned split
    chain_to_split_map = {}
    for split_name, reps in split_reps.items():
        for rep in reps:
            for member_chain in clusters[rep]:
                chain_to_split_map[member_chain] = split_name

    # 4. Execute strict multi-chain anti-leakage mapping (Test > Val > Train)
    print("Step 3.4: Integrating samples with strict hierarchy (Test > Val > Train)...")
    complex_to_chains_map = defaultdict(list)
    for chain_id, info in seq_id_to_source.items():
        complex_to_chains_map[info['complex_id']].append(chain_id)

    final_splits = {'train': set(), 'validation': set(), 'test': set()}
    
    for complex_id, member_chains in complex_to_chains_map.items():
        if not member_chains:
            continue
            
        chain_splits = {chain_to_split_map.get(c) for c in member_chains if c in chain_to_split_map}
        if not chain_splits:
            continue

        # Anti-leakage golden rule hierarchy
        if 'test' in chain_splits:
            final_splits['test'].add(complex_id)
        elif 'validation' in chain_splits:
            final_splits['validation'].add(complex_id)
        else:
            final_splits['train'].add(complex_id)

    # Convert sets to sorted lists
    final_splits_output = {
        'train': sorted(list(final_splits['train'])),
        'validation': sorted(list(final_splits['validation'])),
        'test': sorted(list(final_splits['test']))
    }

    # Save the final results
    with open(FINAL_SPLIT_JSON, 'w') as f:
        json.dump(final_splits_output, f, indent=4)
        
    print("\nSplit generation complete!")
    print(f"Final Train complexes: {len(final_splits_output['train'])}")
    print(f"Final Val complexes:   {len(final_splits_output['validation'])}")
    print(f"Final Test complexes:  {len(final_splits_output['test'])}")
    print(f"Final splits file saved to: {FINAL_SPLIT_JSON}")

if __name__ == "__main__":
    generate_final_splits()

In [ ]:
import json
import os

# --- Path Configuration ---
# 1. Linear pre-training data
PRETRAIN_FASTA = "/home/qiuyk/data/split_workflow/all_sequences.fasta"
PRETRAIN_JSON = "/home/qiuyk/data/split_workflow/for_test/data_splits_optimized_test.json"

# 2. CPCore data (Confirmed paths)
CPCORE_FASTA = "/data/CPSea/CPCore/split_workflow/all_sequences.fasta" 
CPCORE_SPLIT_JSON = "/data/CPSea/CPCore/split_workflow/data_splits.json" 

# 3. Output paths
OUT_DIR = "/data/CPSea/CPCore/leakage_check"
os.makedirs(OUT_DIR, exist_ok=True)

TARGET_DB_FASTA = os.path.join(OUT_DIR, "target_pretrain_train.fasta")
QUERY_DB_FASTA = os.path.join(OUT_DIR, "query_cpcore_val_test.fasta")

def get_pretrain_base_id(header):
    """
    Precise parsing: multichain-6y0b_bio_1_AB-A -> 6y0b_bio_1_AB
    """
    temp = header.replace("multichain-", "")
    if "-" in temp:
        temp = temp.rsplit("-", 1)[0]
    return temp

def get_cpcore_base_id(header):
    """
    Precise parsing for AlphaFold-based CPCore headers.
    Input: CPCore-AF-W0J0Z6-F1_0_263_274_relaxed_relaxed-L
    Output: AF-W0J0Z6-F1_0_263_274_relaxed_relaxed
    """
    # 1. Remove the 'CPCore-' prefix
    temp = header.replace("CPCore-", "")
    
    # 2. Strip the chain ID suffix (e.g., '-L' or '-R')
    # Using rsplit("-", 1) ensures we only split at the LAST hyphen
    if "-" in temp:
        temp = temp.rsplit("-", 1)[0]
        
    return temp

def main():
    print("="*50)
    print(" 🚀 Step 1: Extract FASTA sequences for leakage detection")
    print("="*50)

    # ---------------------------------------------------------
    # Task A: Extract pre-training Train sequences (as Target DB)
    # ---------------------------------------------------------
    with open(PRETRAIN_JSON, 'r') as f:
        pretrain_splits = json.load(f)
    
    pretrain_train_ids = set(pretrain_splits.get("pre-training", {}).get("train", []))
    print(f"[*] Retrieved {len(pretrain_train_ids)} Train IDs from pre-training JSON")

    written_pretrain = 0
    with open(PRETRAIN_FASTA, 'r') as in_f, open(TARGET_DB_FASTA, 'w') as out_f:
        header = ""
        for line in in_f:
            line = line.strip()
            if line.startswith(">"):
                header = line
            else:
                base_id = get_pretrain_base_id(header[1:])
                if base_id in pretrain_train_ids:
                    out_f.write(f"{header}\n{line}\n")
                    written_pretrain += 1
    print(f"[+] Successfully wrote {written_pretrain} pre-training Train sequences -> {TARGET_DB_FASTA}")

    # ---------------------------------------------------------
    # Task B: Extract CPCore Val/Test sequences (as Query DB)
    # ---------------------------------------------------------
    with open(CPCORE_SPLIT_JSON, 'r') as f:
        cpcore_splits = json.load(f)
        
    cpcore_eval_ids = set(cpcore_splits.get("validation", [])) | set(cpcore_splits.get("test", []))
    print(f"\n[*] Retrieved {len(cpcore_eval_ids)} Val/Test IDs from CPCore JSON")

    written_cpcore = 0
    with open(CPCORE_FASTA, 'r') as in_f, open(QUERY_DB_FASTA, 'w') as out_f:
        header = ""
        for line in in_f:
            line = line.strip()
            if line.startswith(">"):
                header = line
            else:
                base_id = get_cpcore_base_id(header[1:])
                # Since we fixed the ID parser, it will now match the JSON keys perfectly.
                # Both Receptor (-R) and Peptide (-L) chains will be extracted.
                if base_id in cpcore_eval_ids:
                    out_f.write(f"{header}\n{line}\n")
                    written_cpcore += 1
                    
    print(f"[+] Successfully wrote {written_cpcore} CPCore evaluation sequences -> {QUERY_DB_FASTA}")
    
    print("\n✅ Sequence extraction complete. Please proceed with the MMseqs2 search command!")

if __name__ == "__main__":
    main()

In [ ]:
# cd /data/CPSea/CPCore/leakage_check

# mmseqs createdb target_pretrain_train.fasta pretrain_target_db

# mmseqs createdb query_cpcore_val_test.fasta cpcore_query_db

# mkdir -p tmp

# mmseqs search cpcore_query_db pretrain_target_db search_out tmp -a --min-seq-id 0.3

# mmseqs convertalis cpcore_query_db pretrain_target_db search_out leakage_report.tsv --format-output query,target,pident,qcov,tcov,evalue

In [ ]:
import json
import pandas as pd
import os

# --- Path Configuration ---
# 1. The report generated by MMseqs2
TSV_FILE = "/data/CPSea/CPCore/leakage_check/leakage_report.tsv"

# 2. Your original CPCore splits
INPUT_JSON = "/data/CPSea/CPCore/split_workflow/data_splits.json"

# 3. The final, absolutely clean splits output
OUTPUT_JSON = "/data/CPSea/CPCore/split_workflow/data_splits_purified.json"

# Threshold: Only consider it a leak if more than 50% of the sequence aligns
# (MMseqs2 already filtered for > 30% sequence identity)
MIN_QCOV = 0.50

def get_cpcore_base_id(header):
    """
    Precise parsing: CPCore-AF-W0J0Z6-F1_0_263_274_relaxed_relaxed-R 
    -> AF-W0J0Z6-F1_0_263_274_relaxed_relaxed
    Must match the exact extraction logic used previously.
    """
    temp = header.replace("CPCore-", "")
    if "-" in temp:
        temp = temp.rsplit("-", 1)[0]
    return temp

def main():
    print("="*50)
    print(" 🛡️ Step 2: Purify Data Splits Based on Leakage Report")
    print("="*50)

    # 1. Load original splits
    print("[*] Loading original CPCore splits...")
    with open(INPUT_JSON, 'r') as f:
        splits = json.load(f)
        
    # 2. Parse the MMseqs2 TSV report
    print(f"[*] Parsing MMseqs2 report: {TSV_FILE}")
    columns = ['query', 'target', 'pident', 'qcov', 'tcov', 'evalue']
    try:
        df = pd.read_csv(TSV_FILE, sep='\t', names=columns)
    except FileNotFoundError:
        print("\n[!] Error: TSV file not found. Did MMseqs2 run successfully?")
        return

    if df.empty:
        print("\n🎉 Incredible! The TSV is totally empty. Zero leakage detected!")
        # If completely clean, just copy original to purified
        leaked_ids = set()
    else:
        # Filter for heavy leakage (significant sequence coverage)
        heavy_leaks = df[df['qcov'] >= MIN_QCOV]
        
        # Extract base complex IDs from the leaked queries
        leaked_ids = set([get_cpcore_base_id(q) for q in heavy_leaks['query']])
        print(f"🚨 ALERT: Found {len(leaked_ids)} Val/Test complexes leaked in pre-training!")

    # 3. Demote leaked samples to Train
    print("\n[*] Executing demotion protocol (Val/Test -> Train)...")
    
    purified_splits = {
        'train': set(splits.get('train', [])), 
        'validation': set(), 
        'test': set()
    }
    
    # Process Validation set
    for vid in splits.get('validation', []):
        if vid in leaked_ids:
            purified_splits['train'].add(vid)  # Demote to train
        else:
            purified_splits['validation'].add(vid)  # Keep in validation
            
    # Process Test set
    for tid in splits.get('test', []):
        if tid in leaked_ids:
            purified_splits['train'].add(tid)  # Demote to train
        else:
            purified_splits['test'].add(tid)  # Keep in test

    # 4. Save the purified JSON
    final_output = {k: sorted(list(v)) for k, v in purified_splits.items()}
    with open(OUTPUT_JSON, 'w') as f:
        json.dump(final_output, f, indent=4)
        
    print("\n" + "="*45)
    print(" ✅ Purification Complete! The ultimate dataset is ready.")
    print("="*45)
    print(f"Original Train: {len(splits.get('train', []))} -> Purified Train: {len(final_output['train'])} (Absorbed leaks)")
    print(f"Original Val  : {len(splits.get('validation', []))} -> Purified Val  : {len(final_output['validation'])}")
    print(f"Original Test : {len(splits.get('test', []))} -> Purified Test : {len(final_output['test'])}")
    print(f"\n[+] Saved pure splits to: {OUTPUT_JSON}")

if __name__ == "__main__":
    main()

In [2]:
import json
import os
from collections import defaultdict
from tqdm import tqdm

# --- Path Configuration ---
WORK_DIR = "/data/CPSea/CPCore/split_workflow/"
INPUT_SPLITS = os.path.join(WORK_DIR, "data_splits_purified.json") 
INPUT_SOURCE_MAP = os.path.join(WORK_DIR, "seq_id_to_source_map.json")
INPUT_CLUSTERS = os.path.join(WORK_DIR, "clusterRes_cluster.tsv")

OUTPUT_SAMPLE_TO_CLUSTER = os.path.join(WORK_DIR, "cpcore_test_sample_to_clusters.json")
OUTPUT_OPTIMIZED_SPLITS = os.path.join(WORK_DIR, "data_splits_purified_optimized.json")
OUTPUT_OPTIMIZED_LIST = os.path.join(WORK_DIR, "cpcore_optimized_test_list.json")

def optimize_test_set():
    print("="*50)
    print(" 🎯 Step 3 (Post-Purification): Greedy Set Cover for Test Set Optimization")
    print("="*50)

    if not os.path.exists(INPUT_SPLITS):
        print(f"Error: {INPUT_SPLITS} not found. Please run the Leakage Purification step first.")
        return

    with open(INPUT_SPLITS, 'r') as f:
        original_splits = json.load(f)
    
    cpcore_test_ids = set(original_splits.get('test', []))
    print(f"[*] Loaded {len(cpcore_test_ids)} purified Test samples.")

    # chain -> representative
    chain_to_rep = {}
    with open(INPUT_CLUSTERS, 'r') as f:
        for line in f:
            rep, member = line.strip().split('\t')
            chain_to_rep[member] = rep
    print(f"[*] Loaded {len(chain_to_rep)} chain-to-cluster mappings.")

    # chain -> complex_id
    with open(INPUT_SOURCE_MAP, 'r') as f:
        seq_id_to_source = json.load(f)

    # Complex IDs -> Clusters
    sample_to_clusters_map = defaultdict(set)
    all_test_clusters = set()
    
    for seq_id, source_info in seq_id_to_source.items():
        complex_id = source_info['complex_id']
        if complex_id in cpcore_test_ids:
            rep = chain_to_rep.get(seq_id)
            if rep:
                sample_to_clusters_map[complex_id].add(rep)
                all_test_clusters.add(rep)

    print(f"[*] Mapping complete. Original test set covers {len(all_test_clusters)} independent clusters.")

    serializable_map = {k: sorted(list(v)) for k, v in sample_to_clusters_map.items()}
    with open(OUTPUT_SAMPLE_TO_CLUSTER, 'w') as f:
        json.dump(serializable_map, f, indent=4)

    # Greedy Set Cover
    print("\n[*] Running Greedy Set Cover Algorithm...")
    remaining_samples_map = {sample: set(clusters) for sample, clusters in serializable_map.items()}
    clusters_to_cover = all_test_clusters.copy()
    optimized_test_set = []

    with tqdm(total=len(clusters_to_cover), desc="Covering Clusters") as pbar:
        while clusters_to_cover and remaining_samples_map:
            best_sample = None
            best_gain = 0

            for sample_id, clusters in remaining_samples_map.items():
                gain = len(clusters.intersection(clusters_to_cover))
                if gain > best_gain:
                    best_gain = gain
                    best_sample = sample_id

            if best_sample is None:
                break

            optimized_test_set.append(best_sample)

            covered_by_best = remaining_samples_map.pop(best_sample)
            newly_covered_count = len(clusters_to_cover.intersection(covered_by_best))
            clusters_to_cover.difference_update(covered_by_best)
            pbar.update(newly_covered_count)

    print("\n" + "="*45)
    print(" ✅ Optimization Complete!")
    print("="*45)
    print(f"Original Test Set Size  : {len(cpcore_test_ids)}")
    print(f"Optimized Test Set Size : {len(optimized_test_set)}")
    print(f"Total Clusters Covered  : {len(all_test_clusters)}")

    new_splits = original_splits.copy()
    new_splits['test'] = sorted(optimized_test_set)

    with open(OUTPUT_OPTIMIZED_SPLITS, 'w') as f:
        json.dump(new_splits, f, indent=4)
    print(f"\n[+] Saved optimized splits to: {OUTPUT_OPTIMIZED_SPLITS}")
    print(f"    (Use this file for training/testing!)")

    with open(OUTPUT_OPTIMIZED_LIST, 'w') as f:
        json.dump(sorted(optimized_test_set), f, indent=4)

if __name__ == "__main__":
    optimize_test_set()

 🎯 Step 3 (Post-Purification): Greedy Set Cover for Test Set Optimization
[*] Loaded 1027 purified Test samples.
[*] Loaded 143734 chain-to-cluster mappings.
[*] Mapping complete. Original test set covers 941 independent clusters.

[*] Running Greedy Set Cover Algorithm...


Covering Clusters: 100%|██████████| 941/941 [00:00<00:00, 10706.98it/s]


 ✅ Optimization Complete!
Original Test Set Size  : 1027
Optimized Test Set Size : 818
Total Clusters Covered  : 941

[+] Saved optimized splits to: /data/CPSea/CPCore/split_workflow/data_splits_purified_optimized.json
    (Use this file for training/testing!)
